# EGFR Experiments: Osimertinib PK Simulation

Simulate daily oral dosing of osimertinib (80 mg) with a half-life of ~48 hours.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '../src')
from rpa_control.style import set_style

set_style()

# --- Parameters ---
dose = 80.0          # mg, standard daily dose
half_life = 48.0     # hours
k_el = np.log(2) / half_life  # elimination rate constant (1/hr)
dosing_interval = 24.0  # hours (once daily)
n_days = 14
T = n_days * 24.0    # total simulation time (hours)
dt = 0.1             # hours

# --- Simulate: dC/dt = -k_el * C, with dose added every 24h ---
n_steps = int(T / dt)
t = np.linspace(0, T, n_steps + 1)
C = np.zeros(n_steps + 1)

for i in range(n_steps):
    # Add dose at the start of each day
    hour = t[i]
    if i == 0 or (int(hour / dosing_interval) > int(t[i-1] / dosing_interval)):
        C[i] += dose
    # First-order elimination
    C[i+1] = C[i] * np.exp(-k_el * dt)

# Steady-state analytics
C_max_ss = dose / (1 - np.exp(-k_el * dosing_interval))
C_min_ss = C_max_ss * np.exp(-k_el * dosing_interval)
print(f'k_el = {k_el:.4f} /hr')
print(f'Steady-state Cmax ~ {C_max_ss:.1f} mg, Cmin ~ {C_min_ss:.1f} mg')

# --- Plot ---
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(t / 24, C, lw=1.5, color='C0')
ax.axhline(C_max_ss, color='C3', ls='--', lw=0.8, alpha=0.6, label=f'Steady-state Cmax ({C_max_ss:.0f} mg)')
ax.axhline(C_min_ss, color='C2', ls='--', lw=0.8, alpha=0.6, label=f'Steady-state Cmin ({C_min_ss:.0f} mg)')
ax.set_xlabel('Time (days)')
ax.set_ylabel('Osimertinib concentration (mg)')
ax.set_title('Osimertinib 80 mg daily — plasma concentration')
ax.legend(fontsize=8)
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
plt.tight_layout()
plt.show()